In [ ]:
# RAG and Agent Evaluation

123


In [2]:
"""
RAG evaluation checks the whole flow together.

This includes:

search
prompt
LLM



Agent evaluation also checks the tool calls.
"""

'\nRAG evaluation checks the whole flow together.\n\nThis includes:\n\nsearch\nprompt\nLLM\n\n\n\nAgent evaluation also checks the tool calls.\n'

In [3]:
"""
LLM as a judge: 

original ansewr and generated answer will be ideally semantically same, 
but the wording will be different so another llm should judge if they are identical.

It is better to aks the llm to reason, instead of just the verdict.
"""

'\nLLM as a judge: \n\noriginal ansewr and generated answer will be ideally semantically same, \nbut the wording will be different so another llm should judge if they are identical.\n\nIt is better to aks the llm to reason, instead of just the verdict.\n'

In [6]:
# Load ground truth data we had so far

import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [7]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I’m a student—do I need the Zoom link?',
 'document': '489dd1c9d9'}

In [9]:
# We still need search function as it is part of RAG
# We need the original (questions and) answers = FAQ

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)



# Create lookup table for the original faq documents 

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [10]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I’m a student—do I need the Zoom link?',
 'document': '489dd1c9d9'}

In [11]:
doc_idx[q["document"]]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [12]:
# In general: llms that are used to evaluate other llms, are called judges.

In [13]:
# Initializing the openai api:

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
# The RAGWithUsage extens RAGBase from module 01,

# and on top tracks usages and how much money spent on evaluation: 
# tracks response usages, and has field usages, and everytime we make a call to the llm we add the costs to costs total

# and also uses the tuned parameters from previous lesson.


from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [ ]:
# Actual question

q["question"]

'How do I join the Office Hours or live workshop if I’m a student—do I need the Zoom link?'

In [ ]:
# Reference answer

doc_idx[q["document"]]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [22]:
# Answer

answer = assistant.rag(
    q["question"]
)

In [23]:
# total cost of all this: 

assistant.total_cost()

0.0019605

In [17]:
# In the next lesson we will run LLM as a judge on all the questions. 

# For now we need to collect the answers to all the questions. 

In [24]:
# Get the original answer from the document ID:

doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [25]:
# Now save both answers in one record:

rag_result = {
    "question": q["question"],  # question
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if I’m a student—do I need the Zoom link?',
 'answer_llm': 'No—you don’t need the Zoom link as a student.\n\nStudents join Office Hours or live workshop sessions via **YouTube Live**, and submit questions through **Slido**. The live video URL is usually posted in the **announcements channel on Telegram and Slack** before the session starts, and you can also watch on the **DataTalksClub YouTube Channel**.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c

In [26]:
# Create a function that processes one ground truth record:

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [27]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the Office Hours or live workshop if I’m a student—do I need the Zoom link?',
 'answer_llm': 'No—students do not get the Zoom link.\n\nYou should join Office Hours or live workshops via:\n- **YouTube Live** for watching\n- **Slido** for submitting questions (the link is pinned in chat when live)\n- The session URL is usually posted in the **announcements channel on Telegram and Slack** before it starts\n\nAlso, don’t post questions in chat, since they may be missed.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very ac

In [28]:
# reset usage: 
assistant.reset_usage()

assistant.total_cost()

0.0

In [29]:
# Processing in parallel: using the same helper as last time.
# we didn't use retry mechanism though because forgot


from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/590 [00:00<?, ?it/s]

In [30]:
# Collect the answer records:

answers = []

for answer_record in results:
    answers.append(answer_record)

In [31]:
# Calculate the total cost:

assistant.total_cost()

0.6458287499999993

In [32]:
# Save the answers:

df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [ ]:
# NOW WE WILL NEXT EVALUATE WITH LLM AS A JUDGE

# we will look at the llm answer and the original ansewr aross the 
# entire doucument and we will see how good is the rag.

In [33]:
# we will use structured output 

# verdict = score
# reasoning 

from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [34]:
# judge instructions prompt 

aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [35]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [36]:
# 

from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [39]:
rec = answers[0]
rec

{'question': 'Is it still possible to join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late. If you want a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [40]:
# now we use this structured retry 

# create formatted prompt -> pass to llm structured retry 

prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the ground truth meaning: late joining is allowed, but certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

In [41]:
# calc_price(usage)
calc_price(usage)

{'input_cost': 0.00022349999999999998,
 'output_cost': 0.000216,
 'total_cost': 0.0004395}

In [43]:
# to run on all recrods, we need of coure put into a function 

def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [44]:
# test 

eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the ground truth meaning: late joining is possible, and certificate eligibility depends on submitting the project while submissions are still open. It is semantically equivalent and accurate.', score='good')

In [45]:
# Run the evaluation on all answers:

def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [46]:
# run in parallel: 

from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/590 [00:00<?, ?it/s]

In [47]:
# split into results and usages again: 

evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [48]:
# Create a dataframe:

df_eval = pd.DataFrame(evaluations)

In [49]:
# Calculate the total cost:

calc_total_price(usages)

0.41897625000000016

In [ ]:
# lets look at the dataframe : and the results 

df_eval.score.value_counts(normalize=True)


# -> the judge doesn't seem strict at all
# -> we should sample some data and check if its ok, otherwise rework the prompt for the judge

# for the project these steps are required

score
good    0.955932
bad     0.044068
Name: proportion, dtype: float64

In [ ]:
# Check the results:

good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")


Good: 564/590 = 95.59%


In [ ]:
# Look at the "bad" cases to understand what went wrong:

df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
1,Can I start the course after it has already be...,74eb249bbf,bad,The AI answer correctly says that you can star...
29,Can I prepare the capstone on my own and then ...,69d122f12e,bad,The AI answer captures the key policy that cap...
33,Is the Capstone project enough to qualify for ...,9f689c185f,bad,The AI answer conflicts with the ground truth....
43,When can I expect this course to be available ...,bd31146b0e,bad,The ground truth gives a specific availability...
57,How do I know when a live session is happening...,d65e05bd7a,bad,The AI answer captures the key idea that live ...


In [57]:
"""
Tip: 

- can also use ai assistant to generate streamlit / webapplication in  react or whatever 
- an application where i can explore this dataset and say if this evaluation is correct or not, 
- because i need to evaluate the judge, and for this i can not use another judge/llm, i need to do this manually.

- then i;ll get something back that shows how many cases the judge was correct or not,
- and then i'll use this information to actually adjust the behavior of my judge. 


"""

"\nTip: \n\n- can also use ai assistant to generate streamlit / webapplication in  react or whatever \n- an application where i can explore this dataset and say if this evaluation is correct or not, \n- because i need to evaluate the judge, and for this i can not use another judge/llm, i need to do this manually.\n\n- then i;ll get something back that shows how many cases the judge was correct or not,\n- and then i'll use this information to actually adjust the behavior of my judge. \n\n\n"

In [58]:
# Save the judged answers:

df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

In [ ]:
# Agent Evaluation

"""
Same approcah can be used to evaluate agents. 

Idea: use agent instead of rag to generate final answers.
(Agent decides to invoke search function one or multiple times.)

We also collect all the tool calls ('trajectories') the agent is making 
and put this into our dataframe, and then evaluating not only the final answers 

but also ask judge to evaluate the tool calls (trajectories).

For that we somehow extract the trajectories and add this task to the judge prompt. 

BASICALLY: IS ANSWER GOOD + IS TRAJECTORY GOOD? -> 2 SCORES.

What is a good trajectory? if the agent goes crazy with the search tool and uses it 10 times, its bad. ..




"""


# TO BE COMPLETED ACCORDING TO https://github.com/DataTalksClub/llm-zoomcamp/blob/main/04-evaluation/lessons/14-agent-evaluation.md

In [59]:
# Answer

answer = assistant.rag(
    "Are there going to the multiple capstone project submission deadlines for the llm zoom camp in 2026? what are the deadline dates?"
)

In [ ]:
answer

# correct answer: https://courses.datatalks.club/llm-zoomcamp-2026/
# 3.8. / 10.8. 

"I don't know."